In [9]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from dotenv import load_dotenv
import sqlite3
import hashlib
import time

load_dotenv()
print("✅ Ready for production patterns!")

✅ Ready for production patterns!


### **Node-Level Caching**

In [2]:
CACHE = {}

class CachedState(TypedDict):
    query: str
    result: str
    cache_hit: bool

def cached_search(state: CachedState) -> dict:
    """Search with caching to avoid redundant API calls."""
    query = state["query"]

    cache_key = hashlib.md5(query.encode()).hexdigest()

    if cache_key in CACHE:
        print(f"✅ Cache HIT for: {query}")
        return {
            "result": CACHE[cache_key],
            "cache_hit": True
        }
    print(f"❌ Cache MISS for: {query}")
    print("   Performing expensive search...")
    time.sleep(1)  # Simulate expensive operation
    
    result = f"Search results for: {query}"

    CACHE[cache_key] = result
    
    return {
        "result": result,
        "cache_hit": False
    }

cache_builder = StateGraph(CachedState)
cache_builder.add_node("search", cached_search)
cache_builder.add_edge(START, "search")
cache_builder.add_edge("search", END)
cached_app = cache_builder.compile()

print("\nTest 1: First call (cache miss)")
result1 = cached_app.invoke({"query": "LangGraph tutorial", "result": "", "cache_hit": False})


Test 1: First call (cache miss)
❌ Cache MISS for: LangGraph tutorial
   Performing expensive search...


In [3]:
print("\nTest 2: Same query (cache hit)")
result2 = cached_app.invoke({"query": "LangGraph tutorial", "result": "", "cache_hit": False})


Test 2: Same query (cache hit)
✅ Cache HIT for: LangGraph tutorial


In [4]:
print(f"\n📊 Cache effectiveness:")
print(f"   First call: {result1['cache_hit']}")
print(f"   Second call: {result2['cache_hit']}")


📊 Cache effectiveness:
   First call: False
   Second call: True


### **Node Caching — Official API**

In [5]:
from langgraph.graph import StateGraph, START, END
from langgraph.cache.memory import InMemoryCache
from typing_extensions import TypedDict
import time

class CacheState(TypedDict):
    query: str
    result: str
    cache_hit: bool

def expensive_lookup(state: CacheState) -> dict:
    """Simulated expensive operation (e.g., embedding + vector search)."""
    print(f"🔄 Computing result for: '{state['query']}' (this is expensive!)")
    time.sleep(0.1)  # Simulate expensive operation
    return {
        "result": f"Result for: {state['query']}",
        "cache_hit": False
    }
cache = InMemoryCache()

builder = StateGraph(CacheState)
builder.add_node("lookup", expensive_lookup)
builder.add_edge(START, "lookup")
builder.add_edge("lookup", END)

# Compile with cache — results of "lookup" node are cached by input hash
app = builder.compile(cache=cache)

In [6]:
# First call — cache miss (runs the function)
print("=== First call (cache miss) ===")
start = time.time()
result1 = app.invoke({"query": "LangGraph features", "result": "", "cache_hit": False})
print(f"Result: {result1['result']} (took {time.time()-start:.2f}s)")

=== First call (cache miss) ===
🔄 Computing result for: 'LangGraph features' (this is expensive!)
Result: Result for: LangGraph features (took 0.10s)


In [7]:
# Second call with same input — cache hit (instant)
print("\n=== Second call (cache hit) ===")
start = time.time()
result2 = app.invoke({"query": "LangGraph features", "result": "", "cache_hit": False})
print(f"Result: {result2['result']} (took {time.time()-start:.4f}s — cached!)")


=== Second call (cache hit) ===
🔄 Computing result for: 'LangGraph features' (this is expensive!)
Result: Result for: LangGraph features (took 0.1019s — cached!)


In [8]:
# Different input — cache miss
print("\n=== Third call (different input, cache miss) ===")
start = time.time()
result3 = app.invoke({"query": "Multi-agent patterns", "result": "", "cache_hit": False})
print(f"Result: {result3['result']} (took {time.time()-start:.2f}s)")


=== Third call (different input, cache miss) ===
🔄 Computing result for: 'Multi-agent patterns' (this is expensive!)
Result: Result for: Multi-agent patterns (took 0.10s)


### **Pre/Post Model Hooks for Guardrails**

In [10]:
class GuardedState(TypedDict):
    messages: Annotated[list, add_messages]
    filtered: bool

BLOCKED_WORDS = ["hack", "exploit", "malicious"]

def pre_model_hook(state: GuardedState) -> dict:
    """Run before LLM call - validate and sanitize input."""
    last_message = state["messages"][-1].content

    print("🔒 Pre-hook: Checking input...")
    
    # Check for blocked content
    if any(word in last_message.lower() for word in BLOCKED_WORDS):
        print("   ⚠️  Blocked content detected!")
        return {
            "filtered": True,
            "messages": [SystemMessage(content="I cannot help with that request.")]
        }
    
    print("   ✅ Input validated")
    return {"filtered": False}

def call_model_with_hooks(state: GuardedState) -> dict:
    """Call LLM with pre/post hooks."""
    # Pre-hook
    pre_result = pre_model_hook(state)

    if pre_result.get("filtered"):
        return pre_result

    llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
    )
    response = llm.invoke(state["messages"])

    # Post-hook
    print("🔒 Post-hook: Checking output...")
    # Could add output filtering here
    print("   ✅ Output validated")

# Build guarded graph
guarded_builder = StateGraph(GuardedState)
guarded_builder.add_node("agent", call_model_with_hooks)
guarded_builder.add_edge(START, "agent")
guarded_builder.add_edge("agent", END)
guarded_app = guarded_builder.compile()

In [11]:
# Test with safe input
print("\nTest 1: Safe input")
result1 = guarded_app.invoke({
    "messages": [HumanMessage(content="Explain Python decorators")],
    "filtered": False
})
print(f"Response: {result1['messages'][-1].content[:100]}...")



Test 1: Safe input
🔒 Pre-hook: Checking input...
   ✅ Input validated
🔒 Post-hook: Checking output...
   ✅ Output validated
Response: Explain Python decorators...


In [12]:
# Test with blocked content
print("\nTest 2: Blocked content")
result2 = guarded_app.invoke({
    "messages": [HumanMessage(content="How to hack a system?")],
    "filtered": False
})
print(f"Response: {result2['messages'][-1].content}")


Test 2: Blocked content
🔒 Pre-hook: Checking input...
   ⚠️  Blocked content detected!
Response: I cannot help with that request.


### **Official Middleware (LangGraph 1.1.x) 🆕**

In [15]:
try:
    from langchain.agents import create_agent
    from langchain.agents.middleware import ModelRetryMiddleware, PIIMiddleware, SummarizationMiddleware
    from langchain_openai import AzureChatOpenAI
    from langchain_core.tools import tool

    @tool
    def search(query: str) -> str:
        """Search for information."""
        return f"Results for: {query}"

    llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
    )
    production_agent = create_agent(
        llm,
        tools=[search],
        system_prompt="You are a helpful research assistant.",
        middleware=[
            ModelRetryMiddleware(
                max_retries=3,
                on_failure="error",  # retry on any error
            ),
            PIIMiddleware("email", strategy="redact", apply_to_input=True
            ),
            SummarizationMiddleware(
                max_tokens=4000,  # Summarize history when it exceeds this
                model=AzureChatOpenAI(
                    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
                    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
                    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
                    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
                    temperature=0.7,
                )
            ),

        ]

    )
    app = production_agent #.compile()
    print("✅ Production agent with middleware created successfully")
    print("Middleware stack: Retry → Content Moderation → Summarization")

except ImportError as e:
    print(f"Note: Middleware requires langchain>=1.2.x: {e}")
    print("Install with: uv pip install langchain>=1.2.15")

✅ Production agent with middleware created successfully
Middleware stack: Retry → Content Moderation → Summarization
